# Complex cashflows (foundation for bonds and loans)

**Prerequisites:** **05** (pricing fundamentals). We map **cashflow engineering** ideas to `bond` JSON (`cashflow_spec`) and price a **linearly amortizing** note.


## Concept

- **Amortization:** bullet, linear paydown, custom schedules.
- **PIK vs cash:** changes outstanding principal paths.
- **Floaters:** caps/floors; **calls/puts** reshape horizon risk.

Deposits remain the simplest pipeline check; bonds carry richer schedules.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext

print("Imports OK (complex cashflows).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

bond_amort = {
    "type": "bond",
    "spec": {
        "id": "AMORT-FIXED",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "accrual_method": "Linear",
        "settlement_days": 2,
        "ex_coupon_days": 0,
        "cashflow_spec": {
            "Amortizing": {
                "base": {
                    "Fixed": {
                        "coupon_type": "Cash",
                        "freq": {"count": 6, "unit": "months"},
                        "dc": "Thirty360",
                        "bdc": "following",
                        "calendar_id": "weekends_only",
                        "end_of_month": False,
                        "payment_lag_days": 0,
                        "rate": "0.04",
                        "stub": "ShortFront",
                    }
                },
                "schedule": {"LinearTo": {"final_notional": {"amount": "200000", "currency": "USD"}}},
            }
        },
        "call_put": None,
        "credit_curve_id": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {},
    },
}
bond_json = json.dumps(bond_amort)
try:
    print("validate prefix:", validate_instrument_json(bond_json)[:180], "…")
except Exception as exc:
    print("validate failed:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
market_json = mc.to_json()
print("market_json chars:", len(market_json))


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(bond_json, market_json, AS_OF_STR, model="discounting")
    vr = ValuationResult.from_json(out)
    print(vr)
except Exception as exc:
    print("price failed:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        bond_json, market_json, AS_OF_STR, model="discounting",
        metrics=["dv01", "duration_mod", "convexity"],
    )
    vr = ValuationResult.from_json(out)
    print("dv01:", vr.get_metric("dv01"))
    print("duration_mod:", vr.get_metric("duration_mod"))
    print("convexity:", vr.get_metric("convexity"))
except Exception as exc:
    print("metrics failed:", type(exc).__name__, exc)


## Takeaways

- **Bond `cashflow_spec`** encodes the same ideas as loan schedules in many systems.
- **Validate first**, then price; schedule errors are easier to read from validation.
- **Deposits** are the minimal alternative when you only need PV off one curve.
